# 📈 Stock Price Prediction Using LSTM
## Long Short-Term Memory Neural Network | Financial Time-Series Forecasting

**Domain:** Time-Series Forecasting / Financial Machine Learning  
**Core Technique:** Deep Learning (LSTM)  
**Data Source:** Yahoo Finance via `yfinance`  

---
### Pipeline Stages
1. Data Acquisition  
2. Feature Engineering (SMA, EMA, RSI, MACD, Bollinger Bands)  
3. Preprocessing (scaling, windowing, chronological split)  
4. LSTM Model Architecture  
5. Training with Callbacks  
6. Evaluation (RMSE, MAE, MAPE, R², DA)  
7. Visualization (7 plots)  
8. 30-Day Future Forecast


## ⚙️ Configuration & Imports

Edit the values in the **Config** cell below to change the stock ticker, date range, or model parameters.
All downstream cells will pick up changes automatically.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, sys, json, pickle, random
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import seaborn as sns
import tensorflow as tf
import yfinance as yf
import ta

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print(f"Keras      : {tf.keras.__version__}")


In [ ]:
# ── USER CONFIG ────────────────────────────────────────────────────────────
TICKER          = "AAPL"        # Yahoo Finance ticker symbol
START_DATE      = "2018-01-01"  # Historical start date
END_DATE        = None          # None = LIVE data up to most recent trading session
SEQUENCE_LENGTH = 60            # Sliding window size (timesteps per sample)
TEST_SPLIT      = 0.20          # Fraction of data reserved for testing
VALIDATION_SPLIT= 0.10          # Fraction of train data used for validation
LSTM_UNITS      = [128, 64]     # Units in each stacked LSTM layer
DROPOUT_RATE    = 0.2           # Dropout rate for regularization
DENSE_UNITS     = 32            # Units in intermediate Dense layer
EPOCHS          = 100           # Maximum training epochs
BATCH_SIZE      = 32            # Mini-batch size
LEARNING_RATE   = 0.001         # Adam optimizer initial learning rate
PATIENCE        = 15            # EarlyStopping patience
LR_PATIENCE     = 7             # ReduceLROnPlateau patience
LR_FACTOR       = 0.5           # LR reduction factor
FORECAST_DAYS   = 30            # Days for recursive future forecast
CONFIDENCE_LEVEL= 0.95          # Statistical confidence for forecast uncertainty bands
                                # (0.90 = 1.645σ, 0.95 = 1.96σ, 0.99 = 2.576σ)
                                # Bands are DERIVED FROM TEST RESIDUALS —
                                # never synthetic or hardcoded percentages.
RANDOM_SEED     = 42

# ── Derived paths ─────────────────────────────────────────────────────────────
BASE_DIR   = os.path.abspath(".")
DATA_DIR   = os.path.join(BASE_DIR, "data", "raw")
MODEL_DIR  = os.path.join(BASE_DIR, "models")
PLOTS_DIR  = os.path.join(BASE_DIR, "outputs", "plots")
REPORT_DIR = os.path.join(BASE_DIR, "outputs", "reports")
for _d in [DATA_DIR, MODEL_DIR, PLOTS_DIR, REPORT_DIR]:
    os.makedirs(_d, exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────────────────────
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

# ── Dark plot theme ───────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0d1117",
    "axes.facecolor"  : "#161b22",
    "text.color"      : "#e6edf3",
    "axes.labelcolor" : "#e6edf3",
    "xtick.color"     : "#e6edf3",
    "ytick.color"     : "#e6edf3",
    "axes.edgecolor"  : "#21262d",
    "grid.color"      : "#21262d",
    "grid.linestyle"  : "--",
    "grid.alpha"      : 0.6,
})

COLORS = {
    "actual" : "#58a6ff",
    "pred"   : "#f78166",
    "train"  : "#3fb950",
    "val"    : "#d29922",
    "future" : "#bc8cff",
    "resid"  : "#ffa657",
}

print(f"Config loaded: {TICKER}  ({START_DATE} to {END_DATE or '<LIVE>'})")
print(f"Output dirs: plots={PLOTS_DIR}")


## 📥 Stage 1 — Data Acquisition

Downloads real historical OHLCV data from **Yahoo Finance** using `yfinance`.

- `auto_adjust=True` adjusts for splits and dividends automatically
- Data is cached to CSV to avoid redundant downloads
- No synthetic data, no fallbacks


In [ ]:
raw_path = os.path.join(DATA_DIR, f"{TICKER}_raw.csv")

if os.path.exists(raw_path):
    raw_df = pd.read_csv(raw_path, index_col=0, parse_dates=True)
    print(f"Loaded from cache: {raw_path}")
else:
    print(f"Downloading {TICKER} from Yahoo Finance (live mode: END_DATE is {END_DATE or 'None — fetching latest data'})...")
    if END_DATE is None:
        raw_df = yf.download(TICKER, start=START_DATE, auto_adjust=True, progress=True)
    else:
        raw_df = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=True, progress=True)
    if isinstance(raw_df.columns, pd.MultiIndex):
        raw_df.columns = raw_df.columns.get_level_values(0)
    raw_df = raw_df[["Open", "High", "Low", "Close", "Volume"]]
    raw_df.dropna(how="all", inplace=True)
    raw_df.sort_index(inplace=True)
    raw_df.index.name = "Date"
    raw_df.to_csv(raw_path)
    print(f"Saved to: {raw_path}")

print(f"\nShape       : {raw_df.shape}")
print(f"Date range  : {raw_df.index[0].date()} to {raw_df.index[-1].date()}")
print(f"Trading days: {len(raw_df):,}")
print(f"\nClose price stats:")
print(raw_df["Close"].describe().round(2))
raw_df.tail()


In [ ]:
# Plot 1: Price History with Volume
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True,
    gridspec_kw={"height_ratios": [3, 1], "hspace": 0.06})

ax1.plot(raw_df.index, raw_df["Close"], color=COLORS["actual"], linewidth=1.4, label="Close Price")
ax1.fill_between(raw_df.index, raw_df["Close"], alpha=0.08, color=COLORS["actual"])
ax1.set_title(f"{TICKER} — Historical Close Price", fontsize=14, fontweight="bold", pad=12)
ax1.set_ylabel("Price (USD)", fontsize=11)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax1.legend(fontsize=10)
ax1.grid(True)

vol_colors = [COLORS["train"] if raw_df["Close"].iloc[i] >= raw_df["Close"].iloc[i-1]
              else COLORS["pred"] for i in range(len(raw_df))]
ax2.bar(raw_df.index, raw_df["Volume"] / 1e6, color=vol_colors, alpha=0.7, width=1.5)
ax2.set_ylabel("Volume (M)", fontsize=10)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "01_raw_price_history.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Plot 1 saved: 01_raw_price_history.png")


## 🔧 Stage 2 — Feature Engineering

Computes 10 technical indicators using the `ta` library and appends them as extra features.

| Feature | Type | Description |
|---|---|---|
| SMA_20 | Trend | 20-day Simple Moving Average |
| EMA_20 | Trend | 20-day Exponential Moving Average |
| RSI_14 | Momentum | 14-period Relative Strength Index |
| MACD | Trend | MACD line (12/26 EMA diff) |
| MACD_Signal | Trend | 9-period signal line |
| MACD_Hist | Momentum | MACD − Signal |
| BB_High | Volatility | Upper Bollinger Band |
| BB_Low | Volatility | Lower Bollinger Band |
| BB_Mid | Trend | Bollinger midline (SMA_20) |
| BB_Width | Volatility | Normalized band width |

These turn raw OHLCV into a **15-feature multivariate** input for the LSTM.


In [ ]:
# Forward-fill missing values (no look-ahead bias)
raw_df = raw_df.ffill().dropna()

df = raw_df.copy()

# Moving Averages
df["SMA_20"] = ta.trend.sma_indicator(df["Close"], window=20)
df["EMA_20"] = ta.trend.ema_indicator(df["Close"], window=20)

# RSI
df["RSI_14"] = ta.momentum.rsi(df["Close"], window=14)

# MACD
macd = ta.trend.MACD(df["Close"], window_fast=12, window_slow=26, window_sign=9)
df["MACD"]        = macd.macd()
df["MACD_Signal"] = macd.macd_signal()
df["MACD_Hist"]   = macd.macd_diff()

# Bollinger Bands
bb = ta.volatility.BollingerBands(df["Close"], window=20, window_dev=2)
df["BB_High"]  = bb.bollinger_hband()
df["BB_Low"]   = bb.bollinger_lband()
df["BB_Mid"]   = bb.bollinger_mavg()
df["BB_Width"] = bb.bollinger_wband()

# Drop look-back NaN rows
df.dropna(inplace=True)

FEATURE_COLS   = list(df.columns)
TARGET_COL     = "Close"
target_col_idx = FEATURE_COLS.index(TARGET_COL)
n_features     = len(FEATURE_COLS)

print(f"Features ({n_features}): {FEATURE_COLS}")
print(f"Target column: '{TARGET_COL}' at index {target_col_idx}")
print(f"Data shape after feature engineering: {df.shape}")
df.tail()


In [ ]:
# Plot 2: Train/Test Split preview
split_idx_preview = int(len(df) * (1 - TEST_SPLIT))
train_preview = df.iloc[:split_idx_preview]
test_preview  = df.iloc[split_idx_preview:]

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(train_preview.index, train_preview["Close"],
        color=COLORS["train"], linewidth=1.2, label=f"Training ({len(train_preview):,} days)")
ax.plot(test_preview.index, test_preview["Close"],
        color=COLORS["pred"], linewidth=1.2, label=f"Test ({len(test_preview):,} days)")
ax.axvspan(train_preview.index[0], train_preview.index[-1], alpha=0.06, color=COLORS["train"])
ax.axvspan(test_preview.index[0], test_preview.index[-1], alpha=0.06, color=COLORS["pred"])
ax.axvline(test_preview.index[0], color="#8b949e", linestyle="--", linewidth=1.2, label="Split Boundary")
ax.set_title(f"{TICKER} — Chronological Train / Test Split (80/20)", fontsize=13, fontweight="bold")
ax.set_ylabel("Close Price (USD)", fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(fontsize=10); ax.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "02_train_test_split.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Plot 2 saved: 02_train_test_split.png")


In [ ]:
# Plot 3: Feature Correlation Heatmap
fig, ax = plt.subplots(figsize=(14, 11), facecolor="#0d1117")
ax.set_facecolor("#0d1117")
corr = df.corr()
cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(corr, ax=ax, cmap=cmap, center=0, annot=True, fmt=".2f",
            annot_kws={"size": 7, "color": "#e6edf3"},
            linewidths=0.5, linecolor="#21262d", square=True,
            cbar_kws={"shrink": 0.75})
ax.set_title("Feature Correlation Heatmap", fontsize=13, fontweight="bold",
             color="#e6edf3", pad=14)
ax.tick_params(colors="#e6edf3", labelsize=9)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "03_feature_correlation.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Plot 3 saved: 03_feature_correlation.png")


## 🔄 Stage 3 — Preprocessing

### Key Design Rules (time-series correctness)

| Rule | Why it matters |
|---|---|
| **Forward-fill only** | No look-ahead bias when filling gaps |
| **Scaler fit on train only** | Prevents data leakage from test period |
| **Chronological split** | Future data must never train the model |
| **No shuffling anywhere** | Temporal order is the signal |


In [ ]:
# Step 1: Chronological train/test split
split_idx = int(len(df) * (1 - TEST_SPLIT))
train_df  = df.iloc[:split_idx].copy()
test_df   = df.iloc[split_idx:].copy()

print(f"Train: {len(train_df):,} rows | {train_df.index[0].date()} to {train_df.index[-1].date()}")
print(f"Test : {len(test_df):,}  rows | {test_df.index[0].date()} to {test_df.index[-1].date()}")


In [ ]:
# Step 2: Fit MinMaxScaler on TRAINING data ONLY
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(train_df)

# Save scaler for later inference
with open(os.path.join(MODEL_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)
print("Scaler saved to:", os.path.join(MODEL_DIR, "scaler.pkl"))

# Step 3: Transform both sets using the SAME scaler
train_scaled = scaler.transform(train_df)
test_scaled  = scaler.transform(test_df)

print(f"train_scaled shape: {train_scaled.shape}")
print(f"test_scaled  shape: {test_scaled.shape}")


In [ ]:
# Step 4: Sliding window sequence generation
# X[i] = data[i : i+60]  (60 days of features)
# y[i] = data[i+60, target_col_idx]  (next day's close price)

def create_sequences(data, seq_len, tgt_idx):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i : i + seq_len])
        y.append(data[i + seq_len, tgt_idx])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_train_full, y_train_full = create_sequences(train_scaled, SEQUENCE_LENGTH, target_col_idx)
X_test,       y_test       = create_sequences(test_scaled,  SEQUENCE_LENGTH, target_col_idx)

# Chronological validation split (last 10% of training sequences)
val_idx = int(len(X_train_full) * (1 - VALIDATION_SPLIT))
X_tr,  X_val = X_train_full[:val_idx], X_train_full[val_idx:]
y_tr,  y_val = y_train_full[:val_idx], y_train_full[val_idx:]

print(f"X_train : {X_tr.shape}   (samples, timesteps, features)")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")


In [ ]:
# Inverse-transform helper (used in Evaluation and Forecast stages)
def inverse_transform(vals, scaler, n_feat, tgt_idx):
    """Reverse MinMax scaling to recover real price values."""
    vals = np.array(vals).flatten()
    dummy = np.zeros((len(vals), n_feat), dtype=np.float32)
    dummy[:, tgt_idx] = vals
    return scaler.inverse_transform(dummy)[:, tgt_idx]


## 🧠 Stage 4 — LSTM Model Architecture

```
Input(60, 15)
  ↓
LSTM(128, return_sequences=True)   ← learns low-level temporal patterns
  ↓
Dropout(0.2)                        ← regularization
  ↓
LSTM(64, return_sequences=False)    ← captures higher-order dependencies
  ↓
Dropout(0.2)
  ↓
Dense(32, relu)                     ← non-linear transformation
  ↓
Dense(1, linear)                    ← next-day close price (regression)
```

**Why LSTM?**  
Plain RNNs suffer from vanishing gradients — they "forget" information from early in the sequence.
LSTM's gated cell state (forget gate, input gate, output gate) carries long-range dependencies
across 60+ timesteps with minimal decay, making it ideal for price sequences.


In [ ]:
model = Sequential(name="LSTM_StockPredictor", layers=[
    Input(shape=(SEQUENCE_LENGTH, n_features)),
    LSTM(LSTM_UNITS[0], return_sequences=True, name="LSTM_1"),
    Dropout(DROPOUT_RATE, name="Dropout_1"),
    LSTM(LSTM_UNITS[1], return_sequences=False, name="LSTM_2"),
    Dropout(DROPOUT_RATE, name="Dropout_2"),
    Dense(DENSE_UNITS, activation="relu", name="Dense_hidden"),
    Dense(1, activation="linear", name="Output"),
])

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="mse",        # Mean Squared Error — standard for regression
    metrics=["mae"]    # Monitor MAE as secondary metric
)

model.summary()
print(f"\nTotal trainable parameters: {model.count_params():,}")


## 🚀 Stage 5 — Training

**Callbacks:**
- `EarlyStopping(patience=15)` — halts training when `val_loss` stops improving; automatically restores best weights  
- `ModelCheckpoint` — saves the best model (lowest `val_loss`) to disk  
- `ReduceLROnPlateau(patience=7, factor=0.5)` — halves learning rate when `val_loss` plateaus

**Training is NOT shuffled** — sequential order must be preserved.


In [ ]:
model_path = os.path.join(MODEL_DIR, "lstm_model.keras")

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,    # Revert to best epoch on stop
        verbose=1
    ),
    ModelCheckpoint(
        filepath=model_path,
        monitor="val_loss",
        save_best_only=True,          # Only save if val_loss improves
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=LR_FACTOR,
        patience=LR_PATIENCE,
        min_lr=1e-7,
        verbose=1
    ),
]

print(f"Training on {len(X_tr):,} samples, validating on {len(X_val):,} samples")
print(f"Max epochs: {EPOCHS}, Batch size: {BATCH_SIZE}\n")

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    shuffle=False,      # NEVER shuffle sequential data
    verbose=1
)

hist_df = pd.DataFrame(history.history)
best_ep = hist_df["val_loss"].idxmin() + 1
print(f"\nBest epoch   : {best_ep} / {len(hist_df)}")
print(f"Best val_loss: {hist_df['val_loss'].min():.6f}")


In [ ]:
# Plot 4: Training & Validation Loss Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
eps = range(1, len(hist_df) + 1)

# Loss (MSE)
ax1.plot(eps, hist_df["loss"],     color=COLORS["actual"], linewidth=1.8, label="Train Loss")
ax1.plot(eps, hist_df["val_loss"], color=COLORS["pred"],   linewidth=1.8, linestyle="--", label="Val Loss")
ax1.axvline(best_ep, color=COLORS["future"], linewidth=1.2, linestyle=":", label=f"Best Epoch ({best_ep})")
ax1.set_title("Loss (MSE)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss (MSE)")
ax1.legend(fontsize=9); ax1.grid(True)

# MAE
ax2.plot(eps, hist_df["mae"],     color=COLORS["actual"], linewidth=1.8, label="Train MAE")
ax2.plot(eps, hist_df["val_mae"], color=COLORS["pred"],   linewidth=1.8, linestyle="--", label="Val MAE")
ax2.set_title("MAE (normalized scale)", fontsize=12, fontweight="bold")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("MAE")
ax2.legend(fontsize=9); ax2.grid(True)

fig.suptitle("LSTM Training History", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "04_training_loss_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Plot 4 saved: 04_training_loss_curves.png")


## 📊 Stage 6 — Evaluation

Predictions are **inverse-transformed** from normalized [0,1] back to real dollar values
before any metric is computed. This ensures all metrics are interpretable in $ units.

| Metric | Description | Goal |
|---|---|---|
| RMSE | Root Mean Squared Error ($) | Minimize |
| MAE  | Mean Absolute Error ($) | Minimize |
| MAPE | Mean Absolute % Error (%) | Minimize |
| R²   | Variance explained (0–1) | Maximize → 1 |
| DA   | Directional Accuracy (%) | Maximize → 100% |


In [ ]:
# Generate predictions on the held-out test set
y_pred_scaled = model.predict(X_test, verbose=0)

# Inverse-transform to real dollar scale
y_pred_real = inverse_transform(y_pred_scaled, scaler, n_features, target_col_idx)
y_true_real = inverse_transform(y_test,         scaler, n_features, target_col_idx)

# Compute metrics
RMSE_val = float(np.sqrt(mean_squared_error(y_true_real, y_pred_real)))
MAE_val  = float(mean_absolute_error(y_true_real, y_pred_real))
MAPE_val = float(np.mean(np.abs((y_true_real - y_pred_real) / y_true_real)) * 100)
R2_val   = float(r2_score(y_true_real, y_pred_real))
delta_t  = np.diff(y_true_real)
delta_p  = np.diff(y_pred_real)
DA_val   = float(np.mean(np.sign(delta_t) == np.sign(delta_p)) * 100)

# Compute residual-derived uncertainty statistics (NEVER synthetic percentages)
residuals_arr = y_true_real - y_pred_real
_ZSCORE_TABLE = {0.90: 1.645, 0.95: 1.96, 0.99: 2.576}
_z = _ZSCORE_TABLE.get(round(CONFIDENCE_LEVEL, 2), 1.96)
residual_stats = {
    "mean":             float(np.mean(residuals_arr)),
    "std":              float(np.std(residuals_arr, ddof=1)),
    "ci_width":         float(_z * np.std(residuals_arr, ddof=1)),
    "confidence_level": float(CONFIDENCE_LEVEL),
    "z_score":          _z,
    "min_residual":     float(np.min(residuals_arr)),
    "max_residual":     float(np.max(residuals_arr)),
    "median_residual":  float(np.median(residuals_arr)),
}

metrics = {
    "ticker"  : TICKER,
    "RMSE"    : round(RMSE_val, 4),
    "MAE"     : round(MAE_val,  4),
    "MAPE"    : round(MAPE_val, 4),
    "R2"      : round(R2_val,   4),
    "DA"      : round(DA_val,   2),
    "residual_stats": residual_stats,
}

# Save metrics
with open(os.path.join(REPORT_DIR, "evaluation_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
flat_row = {k: v for k, v in metrics.items() if k != "residual_stats"}
flat_row.update({f"residual_{k}": v for k, v in residual_stats.items()})
pd.DataFrame([flat_row]).to_csv(os.path.join(REPORT_DIR, "evaluation_metrics.csv"), index=False)

print("=" * 55)
print(f"  MODEL EVALUATION — {TICKER} (Test Set)")
print("=" * 55)
print(f"  Samples evaluated : {len(y_true_real):,}")
print(f"  RMSE              : ${RMSE_val:.4f}")
print(f"  MAE               : ${MAE_val:.4f}")
print(f"  MAPE              : {MAPE_val:.4f}%")
print(f"  R\u00b2                : {R2_val:.4f}")
print(f"  Directional Acc.  : {DA_val:.2f}%")
print(f"  --- Residual-Derived Uncertainty ({CONFIDENCE_LEVEL:.0%}) ---")
print(f"  Residual Mean     : ${residual_stats['mean']:.4f}")
print(f"  Residual Std (\u03c3) : ${residual_stats['std']:.4f}")
print(f"  {_z}\u03c3 CI \u00b1 Width    : ${residual_stats['ci_width']:.4f}")
print("=" * 55)
print(f"\nMetrics saved to: {REPORT_DIR}")


In [ ]:
# Plot 5: Actual vs Predicted
test_dates = test_df.index[SEQUENCE_LENGTH:]

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(test_dates, y_true_real, color=COLORS["actual"], linewidth=1.5, label="Actual Close")
ax.plot(test_dates, y_pred_real, color=COLORS["pred"],   linewidth=1.5, linestyle="--", label="Predicted Close")
err = np.abs(y_true_real - y_pred_real)
ax.fill_between(test_dates, y_pred_real - err, y_pred_real + err,
                alpha=0.12, color=COLORS["pred"], label="Error Band")

title = (f"{TICKER} \u2014 Actual vs. Predicted Close Price (Test Set)\n"
         f"RMSE: ${RMSE_val:.2f}  |  MAE: ${MAE_val:.2f}  |  "
         f"MAPE: {MAPE_val:.2f}%  |  R\u00b2: {R2_val:.4f}  |  DA: {DA_val:.1f}%")
ax.set_title(title, fontsize=12, fontweight="bold", pad=12)
ax.set_ylabel("Price (USD)", fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30)
ax.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "05_actual_vs_predicted.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Plot 5 saved: 05_actual_vs_predicted.png")


In [ ]:
# Plot 6: Residuals
residuals = y_true_real - y_pred_real

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Time-series residuals
bar_colors = [COLORS["train"] if r >= 0 else COLORS["pred"] for r in residuals]
ax1.bar(test_dates, residuals, color=bar_colors, alpha=0.7, width=1.5)
ax1.axhline(0, color="#e6edf3", linewidth=0.8, linestyle="--")
ax1.set_title(f"{TICKER} Residuals Over Time", fontsize=12, fontweight="bold")
ax1.set_ylabel("Actual \u2212 Predicted ($)")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30)
ax1.grid(True)

# Distribution
ax2.hist(residuals, bins=40, color=COLORS["resid"], edgecolor="#0d1117", alpha=0.85)
ax2.axvline(0, color="#e6edf3", linewidth=1, linestyle="--")
ax2.axvline(np.mean(residuals), color=COLORS["future"], linewidth=1.5,
            linestyle="-", label=f"Mean: ${np.mean(residuals):.2f}")
ax2.set_title("Residuals Distribution", fontsize=12, fontweight="bold")
ax2.set_xlabel("Residual ($)"); ax2.set_ylabel("Frequency")
ax2.legend(fontsize=9); ax2.grid(True)

fig.suptitle("Prediction Residual Analysis", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "06_residuals.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Plot 6 saved: 06_residuals.png")


## 🔮 Stage 7 — Recursive 30-Day Future Forecast

**Mechanism (rolling prediction):**
1. Seed the model with the last 60 days of real data
2. Predict next day's price
3. Append prediction to the window (drop oldest day)
4. Repeat `FORECAST_DAYS` times

⚠️ **Note:** Error compounds the further out the forecast extends.
Short horizons (≤30 days) are most reliable for this approach.


In [ ]:
def recursive_forecast(model, last_window, scaler, n_features, tgt_idx, n_days=30):
    """Rolling/recursive multi-step forecast."""
    window = last_window.copy()   # shape: (1, seq_len, n_features)
    preds_scaled = []

    for _ in range(n_days):
        pred = model.predict(window, verbose=0)[0, 0]   # scalar
        preds_scaled.append(pred)

        new_step = window[0, -1, :].copy()
        new_step[tgt_idx] = pred                         # update target col
        window = np.concatenate(
            [window[:, 1:, :], new_step.reshape(1, 1, n_features)],
            axis=1
        )

    return inverse_transform(np.array(preds_scaled), scaler, n_features, tgt_idx)


# Seed window: last 60 rows of the full scaled dataset
full_scaled  = np.vstack([train_scaled, test_scaled])
last_window  = full_scaled[-SEQUENCE_LENGTH:].reshape(1, SEQUENCE_LENGTH, n_features)

future_prices = recursive_forecast(model, last_window, scaler, n_features,
                                   target_col_idx, n_days=FORECAST_DAYS)

# Generate future business-day dates
last_date    = df.index[-1]
future_dates = pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=FORECAST_DAYS)

# Display forecast table
forecast_df = pd.DataFrame({
    "Date"           : future_dates,
    "Predicted_Close": np.round(future_prices, 2)
}).set_index("Date")

print(f"\n{FORECAST_DAYS}-Day Price Forecast for {TICKER}:")
print("=" * 35)
print(forecast_df.to_string())
print("=" * 35)


In [ ]:
# Plot 7: Future Forecast
# Confidence band uses residual-derived dollar-denominated CI (NEVER a hardcoded %)
if residual_stats is None or "ci_width" not in residual_stats:
    raise ValueError(
        "residual_stats with 'ci_width' key is REQUIRED by plot_7 (no fallback). "
        "synthetic/hardcoded percentage bands (e.g. ±2%) are strictly forbidden."
    )
_ci_width = float(residual_stats["ci_width"])
_cl = float(residual_stats.get("confidence_level", 0.95))
_zs = residual_stats.get("z_score", 1.96)
_band_label = f"\u00b1${_ci_width:.2f}  {_cl:.0%} CI (test residuals, {_zs}\u03c3)"

n_ctx      = 120
ctx_dates  = df.index[-n_ctx:]
ctx_prices = df["Close"].values[-n_ctx:]

fig, ax = plt.subplots(figsize=(16, 7))
ax.plot(ctx_dates, ctx_prices,
        color=COLORS["actual"], linewidth=1.5, label="Historical Close")
ax.fill_between(ctx_dates, ctx_prices, alpha=0.07, color=COLORS["actual"])

ax.plot(future_dates, future_prices,
        color=COLORS["future"], linewidth=2.0, linestyle="--",
        marker="o", markersize=3, label=f"{FORECAST_DAYS}-Day Forecast")
ax.fill_between(future_dates,
                future_prices - _ci_width,
                future_prices + _ci_width,
                alpha=0.20, color=COLORS["future"], label=_band_label)

ax.axvline(future_dates[0], color="#8b949e", linewidth=1.0, linestyle=":",
           label="Forecast Start")
ax.set_title(f"{TICKER} \u2014 {FORECAST_DAYS}-Day Future Price Forecast",
             fontsize=14, fontweight="bold", pad=14)
ax.set_ylabel("Price (USD)", fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
plt.xticks(rotation=30)
ax.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "07_future_forecast.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Plot 7 saved: 07_future_forecast.png")
print(f"Plot 7 confidence band : {_band_label}")


## ✅ Final Summary & Deliverables

All pipeline stages complete. Below is the consolidated results report.


In [ ]:
# Final consolidated report
print("\n" + "=" * 60)
print(f"  STOCK PRICE PREDICTION — FINAL REPORT")
print(f"  Ticker    : {TICKER}")
print(f"  Period    : {START_DATE} to {END_DATE}")
print(f"  Features  : {n_features} (OHLCV + 10 technical indicators)")
print("=" * 60)
print(f"  RMSE              : ${RMSE_val:.4f}")
print(f"  MAE               : ${MAE_val:.4f}")
print(f"  MAPE              : {MAPE_val:.4f}%")
print(f"  R\u00b2                : {R2_val:.4f}")
print(f"  Directional Acc.  : {DA_val:.2f}%")
print("=" * 60)
print(f"\nDeliverables:")
print(f"  Model     : {model_path}")
print(f"  Scaler    : {os.path.join(MODEL_DIR, 'scaler.pkl')}")
print(f"  Metrics   : {os.path.join(REPORT_DIR, 'evaluation_metrics.json')}")
print(f"  Plots (7) : {PLOTS_DIR}")

import glob
plot_files = sorted(glob.glob(os.path.join(PLOTS_DIR, "*.png")))
for pf in plot_files:
    print(f"    - {os.path.basename(pf)}")
